<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####1. How to create Spark Session

-- This option creates a session using the builder but the next option (below) uses a pre-cretaed version which needs less code

from pyspark.sql import SparkSession

spark_session = SparkSession.builder.getOrCreate()

spark_session.version

####2. What is pre-created Spark Session

In [0]:
#This below is a pre-created spark session. It's used when you want t avoid creating the spark session with teh builder. "spark" is teh precreated session
# We will use this to create a spark session from here onwards
spark.version

'4.1.0'

####3. READ Employee csv
##### FOOL(Format, Option, Option, Load)
spark.read.format().option().option().load()

How to use SparkSession to read a csv file

##### At a first glance some data types would nneed to be fixed becasue of a bad schema inference

In [0]:
raw_emp_df= (
              spark.read.format('csv')
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .load("/Volumes/dev/spark_db/datasets/spark_programming/data/employee.csv")
)

raw_emp_df.display()

employee_id,Name,departmentid,role,salary,startdate,enddate
1,John Smith,1,Sales Associate,60000,2020-01-15,null
2,Sarah Johnson,1,Sales Manager,65000,2019-06-20,null
3,Michael Brown,2,Software Engineer,75000,2018-03-10,null
4,Emily White,2,Software Manager,70000,2021-02-14,null
5,David Lee,3,HR Specialist,80000,2017-11-25,null
6,Jennifer Davis,3,HR Manager,78000,2019-09-01,2023-03-30
7,Robert Wilson,null,Sales Associate,55000,2022-04-12,null
8,Lisa Anderson,4,Marketing Specialist,72000,2020-07-08,null
9,James Taylor,4,Marketing Manager,71000,2021-01-20,null
10,Mary Martinez,null,Sales Associate,58000,2022-05-15,null


#### Fix "departmentid" and "enddate" columns type to "Int" and "date" respectively.
After reading the data we noticed that "departmentid" and "enddate" are inferred incorrectly, both as "String" but it should be:

departmentid ="Int"  and enddate = "date" so this needs to be fixed.


#### From PySpark use either withColumn() or withColumns():

1 withColumn() to add a column or replacing the existing column that has the same name. 

2 withColumns() to add multiple columns or replacing the existing columns that have the same names.

doc: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html



In [0]:
from pyspark.sql.functions import expr, col
""" 
expr(str)          - Parses the expression string into the column that it represents
col(col)           - Returns a Column based on the given column name.
column(col)        - Returns a Column based on the given column name.
lit(col)           - Creates a Column of literal value.
try_cast(dataType) - A special version of cast that performs the same operation, but returns a NULL value instead of raising an error if 
                     the invoke method throws exception.
"""

emp_df = raw_emp_df.withColumns({
    'departmentid': expr("try_cast(departmentid as int)")
    ,'enddate': expr("try_to_date(enddate, 'M/d/yyyy')")
})

emp_df.display()

employee_id,Name,departmentid,role,salary,startdate,enddate
1,John Smith,1,Sales Associate,60000,2020-01-15,null
2,Sarah Johnson,1,Sales Manager,65000,2019-06-20,null
3,Michael Brown,2,Software Engineer,75000,2018-03-10,null
4,Emily White,2,Software Manager,70000,2021-02-14,null
5,David Lee,3,HR Specialist,80000,2017-11-25,null
6,Jennifer Davis,3,HR Manager,78000,2019-09-01,null
7,Robert Wilson,null,Sales Associate,55000,2022-04-12,null
8,Lisa Anderson,4,Marketing Specialist,72000,2020-07-08,null
9,James Taylor,4,Marketing Manager,71000,2021-01-20,null
10,Mary Martinez,null,Sales Associate,58000,2022-05-15,null


#### Load data into the employee table

In [0]:
emp_df.write.mode("overwrite").saveAsTable("dev.spark_db.employee")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-182547903376616>, line 1
----> 1 emp_df.write.mode("overwrite").saveAsTable("dev.spark_db.employee")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1589, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1587     req.user_context.user_id = self._user_id
   1588 self._set_command_in_plan(req.plan, command)
-> 


#### Query the table

In [0]:
%sql

SELECT count(*) FROM dev.spark_db.employee  -- count all records

--df = spark.table("dev.spark_db.employee")
--df.display()

count(*)
108


####3. Department table 
How to use SparkSession to read a csv file

##### At a first glance data looks fine

In [0]:
raw_dep_df= (
              spark.read.format('csv')
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .load("/Volumes/dev/spark_db/datasets/spark_programming/data/department.csv")
)

##### Data is fine so no fixes needed.


In [0]:
dep_df= raw_dep_df
dep_df.display()

##### Load data into department table

In [0]:
dep_df.write.mode("overwrite").saveAsTable("dev.spark_db.department")

#### Query the table

In [0]:
%sql

SELECT count(*) FROM dev.spark_db.department  -- count all records

--df = spark.table("dev.spark_db.employee")
--df.display()

&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>
